# Notebook 01: Khám phá và Kiểm kê Dữ liệu (Data Exploration & Audit)

**Đề tài:** Dự đoán khách hàng rời bỏ dịch vụ viễn thông (Telco Customer Churn).

**Mục tiêu của notebook:**
1. Kiểm kê chất lượng dữ liệu ban đầu trên toàn bộ 7.043 bản ghi nguồn (kiểm tra thiếu, rỗng, trùng lặp, logic nghiệp vụ).
2. Chuẩn hóa tiền phân chia (pre-split cleaning) và xác nhận phân chia phân tầng 80/20 kèm 5-fold CV.
3. **Thực hiện Phân tích Khám phá Dữ liệu (EDA) NGHIÊM NGẶT CHỈ TRÊN TẬP PHÁT TRIỂN (N = 5.634)** nhằm ngăn chặn hoàn toàn rò rỉ thông tin sang tập kiểm thử (Data Leakage).
4. Đưa ra 7 nhận định và phân tích chuyên sâu trả lời đầy đủ các câu hỏi nghiên cứu của đề tài.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn thư viện nội bộ
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from telco_churn.data import load_raw_data, audit_dataset, create_splits
from telco_churn.schema import validate_records

# Cấu hình giao diện biểu đồ
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "sans-serif"]
plt.rcParams["font.size"] = 10

## 1. Kiểm kê và Đánh giá Chất lượng Dữ liệu Nguồn (Audit Raw Data)

Đọc tệp dữ liệu gốc từ `telco-customer-churn-on-icp4d/data/Telco-Customer-Churn.csv`, tính toán mã băm SHA-256 để đảm bảo tính bất biến, thống kê các giá trị rỗng/khoảng trắng, trùng lặp và xung đột nghiệp vụ.

In [ ]:
data_path = os.path.join("..", "telco-customer-churn-on-icp4d", "data", "Telco-Customer-Churn.csv")
df_raw = load_raw_data(data_path)
audit_results = audit_dataset(df_raw, source_file_path=data_path)

print(f"File SHA-256: {audit_results['file_hash']}")
print(f"Kích thước dữ liệu: {audit_results['total_rows']} dòng, {audit_results['total_cols']} cột")
print(f"Phân bố nhãn Churn: {audit_results['target_dist']}")
print(f"Số dòng TotalCharges để trống: {audit_results['tc_blank_count']}")
print(f"Trùng lặp toàn bộ dòng: {audit_results['total_duplicate_rows']}")
print(f"Trùng lặp customerID: {audit_results['id_duplicates']}")
print(f"Xung đột dịch vụ thoại: {audit_results['phone_conflicts']}")
print(f"Xung đột dịch vụ Internet: {audit_results['internet_conflicts']}")

### Phân tích 11 dòng có `TotalCharges` để trống

Kiểm tra xem 11 bản ghi có `TotalCharges` mang chuỗi khoảng trắng có điểm chung gì.

In [ ]:
tc_blank = df_raw[df_raw["TotalCharges"].str.strip() == ""]
tc_blank[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

> **Kết luận kiểm kê:**
- Cả 11 khách hàng này đều có `tenure = 0` và `Churn = No`. Đây là các khách hàng vừa ký hợp đồng mới trong tháng, chưa đến kỳ phát sinh hóa đơn đầu tiên.
- Phương án xử lý: Chuyển khoảng trắng thành `NaN`. Không loại bỏ 11 bản ghi này mà sử dụng `SimpleImputer(strategy='median')` được học độc lập trên từng training fold để điền giá trị thiếu.

## 2. Tiền xử lý Pre-split và Khóa Tập Kiểm thử (Split & Manifest)

- Áp dụng hàm chuẩn hóa và kiểm tra schema định trước `validate_records`.
- Chia phân tầng 80% phát triển (5.634 dòng) và 20% kiểm thử (1.409 dòng).
- Tạo 5-fold Stratified K-Fold trên tập phát triển và lưu manifest truy vết.

In [ ]:
valid_df, errors = validate_records(df_raw, mode="training")
print(f"Số dòng hợp lệ sau chuẩn hóa: {len(valid_df):,} / {len(df_raw):,}")
print(f"Số lỗi/cảnh báo ghi nhận: {len(errors)}")

manifest_path = os.path.join("..", "data", "splits", "split_manifest.csv")
dev_df, test_df, manifest_df = create_splits(
    df=valid_df,
    test_size=0.2,
    random_state=42,
    n_splits=5,
    output_manifest_path=manifest_path,
    source_file_path=data_path,
    cleaning_version="v1"
)

dev_df["Churn"] = dev_df["Churn"].astype(int)
test_df["Churn"] = test_df["Churn"].astype(int)
print(f"Tập phát triển (Dev): {len(dev_df):,} dòng (Tỷ lệ Churn: {dev_df['Churn'].mean():.2%})")
print(f"Tập kiểm thử (Test): {len(test_df):,} dòng (Tỷ lệ Churn: {test_df['Churn'].mean():.2%})")

## 3. Khám phá Dữ liệu (EDA) trên Tập Phát triển (N = 5.634)

### Câu hỏi 1: Nhãn mục tiêu mất cân bằng như thế nào? Tại sao Accuracy là chưa đủ?

In [ ]:
plt.figure(figsize=(6, 4.5))
counts = dev_df["Churn"].value_counts().sort_index()
labels = ["Không rời bỏ (0)", "Rời bỏ (1)"]
colors = ["#2b5c8f", "#d95f02"]
bars = plt.bar(labels, counts.values, color=colors, width=0.5, edgecolor="black", alpha=0.85)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, h + 50, f"{h:,} ({h/len(dev_df)*100:.1f}%)", ha="center", fontweight="bold")
plt.title(f"Phân bố nhãn mục tiêu Churn (Tập phát triển N = {len(dev_df):,})", fontweight="bold")
plt.ylabel("Số lượng khách hàng")
plt.ylim(0, max(counts.values) * 1.15)
plt.show()

**Nhận xét (Câu hỏi 1):**
1. Tập phát triển ghi nhận 4.139 khách hàng giữ dịch vụ (73,46%) và 1.495 khách hàng rời bỏ (26,54%), tỷ lệ mất cân bằng xấp xỉ 1:2,77.
2. Một mô hình cơ sở ngây thơ (dummy baseline) dự đoán toàn bộ là lớp đa số (không rời bỏ) vẫn có thể đạt độ chính xác (Accuracy) lên tới 73,46%, nhưng hoàn toàn vô dụng vì bỏ sót 100% khách hàng rời bỏ.
3. Do đó, Accuracy không phải là thước đo phù hợp; hệ thống cần ưu tiên tối ưu và theo dõi F1-Score của lớp rời bỏ (lớp 1), Recall, Precision và diện tích dưới đường cong ROC-AUC.

### Câu hỏi 2: Thâm niên sử dụng (tenure) có liên hệ như thế nào với nguy cơ rời bỏ?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(data=dev_df, x="tenure", hue="Churn", kde=True, bins=36, palette={0: "#2b5c8f", 1: "#d95f02"}, ax=ax1, element="step", common_norm=False)
ax1.set_title("Phân bố thâm niên theo Churn", fontweight="bold")
ax1.legend(["Rời bỏ (1)", "Không rời bỏ (0)"], title="Churn")

sns.boxplot(data=dev_df, x="Churn", y="tenure", hue="Churn", legend=False, palette={0: "#2b5c8f", 1: "#d95f02"}, ax=ax2, width=0.45)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(["Không rời bỏ (0)", "Rời bỏ (1)"])
ax2.set_title("Boxplot thâm niên theo Churn", fontweight="bold")
plt.tight_layout()
plt.show()

**Nhận xét (Câu hỏi 2):**
1. Nhóm khách hàng rời bỏ có trung vị thâm niên rất thấp (chỉ khoảng 10 tháng), tập trung áp đảo ở khoảng 1–5 tháng đầu tiên sử dụng dịch vụ.
2. Ngược lại, nhóm khách hàng không rời bỏ có trung vị thâm niên cao (khoảng 38 tháng), phân bố trải đều và tăng vọt ở nhóm thâm niên lâu năm (60–72 tháng).
3. Điều này cho thấy mối liên hệ nghịch biến mạnh mẽ: khách hàng càng mới thì nguy cơ rời bỏ càng cao; tuy nhiên, đây là mối tương quan quan sát được trong dữ liệu chứ không khẳng định quan hệ nhân quả trực tiếp (cần kiểm tra thêm các yếu tố trải nghiệm ban đầu).

### Câu hỏi 3: Loại hợp đồng (Contract) liên quan thế nào đến tỷ lệ rời bỏ?

In [ ]:
contract_stats = dev_df.groupby("Contract")["Churn"].agg(total_count="count", churn_count="sum", churn_rate="mean").reindex(["Month-to-month", "One year", "Two year"])
fig, ax1 = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(contract_stats))
width = 0.35
ax1.bar(x, contract_stats["total_count"], width, label="Tổng số khách", color="#6baed6", edgecolor="black", alpha=0.8)
ax1.bar(x, contract_stats["churn_count"], width, label="Khách rời bỏ", color="#d95f02", edgecolor="black", alpha=0.9)
ax1.set_xticks(x)
ax1.set_xticklabels(contract_stats.index, fontweight="bold")
ax1.set_ylabel("Số lượng khách hàng")

ax2 = ax1.twinx()
ax2.plot(x, contract_stats["churn_rate"] * 100, color="#b30000", marker="o", linewidth=2.5, markersize=8, label="Tỷ lệ Churn (%)")
ax2.set_ylabel("Tỷ lệ Churn (%)", color="#b30000")
ax2.set_ylim(0, 60)
for i, r in enumerate(contract_stats["churn_rate"]):
    ax2.text(i, r * 100 + 2.5, f"{r*100:.1f}%", ha="center", fontweight="bold", color="#b30000")
plt.title("Tỷ lệ rời bỏ và quy mô mẫu theo loại hợp đồng (Contract)", fontweight="bold")
plt.tight_layout()
plt.show()

**Nhận xét (Câu hỏi 3):**
1. Hợp đồng từng tháng (Month-to-month) chiếm quy mô lớn nhất (3.086 mẫu) và có tỷ lệ rời bỏ đặc biệt cao lên tới 42,7%.
2. Hợp đồng dài hạn 1 năm và 2 năm có tỷ lệ rời bỏ giảm mạnh lần lượt xuống chỉ còn 11,0% và 2,9%.
3. Quy mô mẫu ở cả 3 nhóm đều rất lớn (tối thiểu hơn 1.300 mẫu/nhóm), do đó sự khác biệt về tỷ lệ rời bỏ có độ tin cậy thống kê cao chứ không phải do sai số mẫu nhỏ.

### Câu hỏi 4: Chi phí (MonthlyCharges, TotalCharges) có sự khác biệt giữa hai nhóm không?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
sns.kdeplot(data=dev_df, x="MonthlyCharges", hue="Churn", common_norm=False, palette={0: "#2b5c8f", 1: "#d95f02"}, fill=True, alpha=0.3, ax=ax1)
ax1.set_title("Mật độ cước hàng tháng (MonthlyCharges)", fontweight="bold")
ax1.legend(["Rời bỏ (1)", "Không rời bỏ (0)"], title="Churn")

sns.boxplot(data=dev_df, x="Churn", y="TotalCharges", hue="Churn", legend=False, palette={0: "#2b5c8f", 1: "#d95f02"}, ax=ax2, width=0.45)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(["Không rời bỏ (0)", "Rời bỏ (1)"])
ax2.set_title("Boxplot tổng chi phí tích lũy (TotalCharges)", fontweight="bold")
plt.tight_layout()
plt.show()

**Nhận xét (Câu hỏi 4):**
1. `MonthlyCharges`: Nhóm khách hàng rời bỏ có đỉnh mật độ tập trung ở phân khúc phí cao (70 – 100 USD/tháng), trong khi nhóm ở lại tập trung nhiều ở phân khúc phí thấp (khoảng 20 USD/tháng - dịch vụ thoại cơ bản).
2. `TotalCharges`: Nhóm rời bỏ có tổng chi phí lũy kế thấp hơn rõ rệt (trung vị khoảng 700 USD so với 1.700 USD của nhóm ở lại), điều này hoàn toàn nhất quán với kết quả thâm niên ngắn ở Câu hỏi 2.
3. Không phát hiện giá trị âm hay ngoại lai hỏng; đuôi dài ở `TotalCharges` phản ánh đúng khách hàng thâm niên 5-6 năm có cước tích lũy cao.

### Câu hỏi 5: Các dịch vụ bổ sung và phương thức thanh toán có tỷ lệ rời bỏ khác nhau không?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
cols = ["InternetService", "TechSupport", "PaymentMethod"]
titles = ["Dịch vụ Internet", "Hỗ trợ kỹ thuật (TechSupport)", "Phương thức thanh toán"]
for ax, col, title in zip(axes, cols, titles):
    stats = dev_df.groupby(col)["Churn"].agg(total="count", rate="mean").sort_values(by="rate", ascending=False)
    bars = ax.bar(range(len(stats)), stats["rate"] * 100, color="#d95f02", edgecolor="black", alpha=0.8, width=0.5)
    ax.set_xticks(range(len(stats)))
    ax.set_xticklabels(stats.index, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("Tỷ lệ Churn (%)")
    ax.set_ylim(0, 55)
    ax.set_title(title, fontweight="bold", fontsize=10)
    for i, bar in enumerate(bars):
        ax.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() + 1.2, f"{stats['rate'].iloc[i]*100:.1f}%", ha="center", fontsize=8.5, fontweight="bold")
plt.tight_layout()
plt.show()

**Nhận xét (Câu hỏi 5):**
1. Dịch vụ Cáp quang (Fiber optic) có tỷ lệ rời bỏ lên tới 41,8%, cao hơn rất nhiều so với DSL (19,0%) và Không dùng Internet (7,4%). Đây là giả thuyết cần kiểm tra về giá cước cáp quang cao và kỳ vọng chất lượng đường truyền.
2. Khách hàng không có gói hỗ trợ kỹ thuật (`TechSupport = No`) có tỷ lệ rời bỏ lên đến 41,5%, gấp gần 3 lần nhóm có hỗ trợ kỹ thuật (15,2%).
3. Phương thức thanh toán séc điện tử (`Electronic check`) có tỷ lệ rời bỏ lên tới 45,3%, trong khi các hình thức thanh toán tự động (thẻ tín dụng/chuyển khoản) duy trì tỷ lệ rời bỏ thấp ổn định ở mức 15-16%.

### Câu hỏi 6: Các biến số liên tục có tương quan tuyến tính mạnh không?

In [ ]:
plt.figure(figsize=(6, 4.5))
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
corr = dev_df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=1)
plt.title("Ma trận tương quan Pearson giữa các biến số liên tục", fontweight="bold")
plt.tight_layout()
plt.show()

**Nhận xét (Câu hỏi 6):**
1. `tenure` và `TotalCharges` có tương quan dương rất mạnh ($r = 0,826$), điều này hoàn toàn hợp lý về mặt nghiệp vụ vì khách hàng ở lại càng nhiều tháng thì tổng số tiền tích lũy càng lớn.
2. `MonthlyCharges` và `TotalCharges` cũng có tương quan dương đáng kể ($r = 0,651$).
3. Lưu ý: Sự đa cộng tuyến giữa `tenure` và `TotalCharges` có thể ảnh hưởng đến độ ổn định trọng số của mô hình hồi quy tuyến tính/logistic (cần điều chuẩn hóa L2 Ridge), nhưng không làm giảm khả năng phân nhánh của cây quyết định và rừng ngẫu nhiên.

### Câu hỏi 7: Chất lượng dữ liệu sau làm sạch và số mẫu trước/sau xử lý

In [ ]:
cleaning_summary = pd.DataFrame([
    {"Giai đoạn": "Dữ liệu CSV gốc", "Số dòng": len(df_raw), "Số cột": len(df_raw.columns), "Ghi chú": "Chứa 11 ô TotalCharges rỗng"},
    {"Giai đoạn": "Sau tiền xử lý pre-split", "Số dòng": len(valid_df), "Số cột": len(valid_df.columns), "Ghi chú": "Bảo toàn 100% mẫu, chuyển rỗng thành NaN"},
    {"Giai đoạn": "Tập phát triển (Dev 80%)", "Số dòng": len(dev_df), "Số cột": len(dev_df.columns), "Ghi chú": "Dùng cho EDA và 5-fold CV (tỷ lệ Churn: 26.54%)"},
    {"Giai đoạn": "Tập kiểm thử (Test 20%)", "Số dòng": len(test_df), "Số cột": len(test_df.columns), "Ghi chú": "Khóa độc lập, không tham gia huấn luyện/tuning"},
])
cleaning_summary

**Nhận xét (Câu hỏi 7):**
1. Quy trình làm sạch thành công bảo toàn 100% mẫu (7.043 / 7.043 bản ghi), không bị hao hụt dữ liệu do xóa dòng.
2. Việc chia tách 80/20 phân tầng đảm bảo phân bố nhãn Churn ở cả hai tập bằng nhau hoàn hảo (26,54%).
3. Toàn bộ thông tin kiểm thử được cô lập nghiêm ngặt, sẵn sàng chuyển giao cho các thành viên xây dựng mô hình ở Plan 03.